In [1]:
import torch
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset,random_split
import matplotlib.pyplot as plt
import torch.optim as optim
import numpy as np
from timm.models.vision_transformer import VisionTransformer
import timm
import os
from tqdm import tqdm
from PIL import Image
from torch.utils.data import ConcatDataset
import torch
from torchvision import datasets
import timm
import copy
import torch.nn as nn
import torch.optim as optim

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [2]:
def create_mae_model_from_timm(new_depth=6):
    # Step 1: Create ViT encoder with same architecture, without downloading
    encoder = timm.create_model(
        'vit_base_patch16_224',
        pretrained=False,  # Prevents downloading
        img_size=224,
        patch_size=16,
        embed_dim=768,
        depth=new_depth,
        num_heads=12,
        mlp_ratio=4.0
    )

    # Step 2: Load local checkpoint (make sure it's compatible with your encoder config)
    ckpt_path = '../models/vit_base_patch16_224.pth'
    state_dict = torch.load(ckpt_path, map_location='cpu')
    
    # If needed, remove "module." prefix (some checkpoints have it)
    if any(k.startswith("module.") for k in state_dict.keys()):
        state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}

    # Load weights
    missing_keys, unexpected_keys = encoder.load_state_dict(state_dict, strict=False)
    print("✅ Loaded encoder weights.")
    print(f"Missing keys: {len(missing_keys)} | Unexpected keys: {len(unexpected_keys)}")

    return encoder


class SimpleMAE(torch.nn.Module):
    def __init__(
        self, 
        encoder,
        decoder_embed_dim=512,
        decoder_depth=8,
        decoder_num_heads=16,
        mask_ratio=0.75,
        norm_pix_loss=True
    ):
        super().__init__()
        self.encoder = encoder
        self.img_size = encoder.patch_embed.img_size[0]
        self.patch_size = encoder.patch_embed.patch_size[0]
        self.embed_dim = encoder.embed_dim
        self.num_patches = (self.img_size // self.patch_size) ** 2
        self.mask_token = torch.nn.Parameter(torch.zeros(1, 1, decoder_embed_dim))
        self.decoder_embed = torch.nn.Linear(self.embed_dim, decoder_embed_dim, bias=True)
        self.decoder_pos_embed = torch.nn.Parameter(torch.zeros(1, self.num_patches + 1, decoder_embed_dim))
        self.decoder_blocks = torch.nn.ModuleList([
            timm.models.vision_transformer.Block(
                decoder_embed_dim, decoder_num_heads, 4.0
            ) for _ in range(decoder_depth)
        ])
        
        self.decoder_norm = torch.nn.LayerNorm(decoder_embed_dim)
        self.decoder_pred = torch.nn.Linear(
            decoder_embed_dim, self.patch_size**2 * 3, bias=True
        )
        
        self.mask_ratio = mask_ratio
        self.norm_pix_loss = norm_pix_loss
        self.initialize_weights()
    
    def initialize_weights(self):
        torch.nn.init.normal_(self.mask_token, std=0.02)
        torch.nn.init.normal_(self.decoder_pos_embed, std=0.02)
        torch.nn.init.xavier_uniform_(self.decoder_embed.weight)
        torch.nn.init.zeros_(self.decoder_embed.bias)
        torch.nn.init.xavier_uniform_(self.decoder_pred.weight)
        torch.nn.init.zeros_(self.decoder_pred.bias)
    
    def random_masking(self, x, mask_ratio):
        N, L, D = x.shape
        len_keep = int(L * (1 - mask_ratio))
        noise = torch.rand(N, L, device=x.device)
        ids_shuffle = torch.argsort(noise, dim=1)
        ids_restore = torch.argsort(ids_shuffle, dim=1)
        ids_keep = ids_shuffle[:, :len_keep]
        x_masked = torch.gather(
            x, dim=1, 
            index=ids_keep.unsqueeze(-1).repeat(1, 1, D)
        )
        mask = torch.ones([N, L], device=x.device)
        mask[:, :len_keep] = 0
        mask = torch.gather(mask, dim=1, index=ids_restore)
        
        return x_masked, mask, ids_restore
    
    def forward_encoder(self, x, mask_ratio):
        patches = self.encoder.patch_embed(x)
        patches = patches + self.encoder.pos_embed[:, 1:, :]
        patches_masked, mask, ids_restore = self.random_masking(patches, mask_ratio)
        cls_token = self.encoder.cls_token + self.encoder.pos_embed[:, :1, :]
        cls_tokens = cls_token.expand(patches_masked.shape[0], -1, -1)
        x = torch.cat((cls_tokens, patches_masked), dim=1)
        for blk in self.encoder.blocks:
            x = blk(x)
        x = self.encoder.norm(x)
        
        return x, mask, ids_restore
    
    def forward_decoder(self, x, ids_restore):
        x = self.decoder_embed(x)
        mask_tokens = self.mask_token.repeat(
            x.shape[0], ids_restore.shape[1] + 1 - x.shape[1], 1
        )
        x_ = x[:, 1:, :]
        x_ = torch.cat([x_, mask_tokens], dim=1)
        x_ = torch.gather(
            x_, dim=1,
            index=ids_restore.unsqueeze(-1).repeat(1, 1, x.shape[2])
        )
        x = torch.cat([x[:, :1, :], x_], dim=1)
        x = x + self.decoder_pos_embed
        for blk in self.decoder_blocks:
            x = blk(x)
        x = self.decoder_norm(x)
        x = self.decoder_pred(x)
        x = x[:, 1:, :]
        
        return x
    
    def patchify(self, imgs):
        p = self.patch_size
        h = w = self.img_size // p
        x = imgs.reshape(shape=(imgs.shape[0], 3, h, p, w, p))
        x = torch.einsum('nchpwq->nhwpqc', x)
        x = x.reshape(shape=(imgs.shape[0], h * w, p**2 * 3))
        
        return x
    
    def unpatchify(self, x):
        p = self.patch_size
        h = w = int(x.shape[1]**0.5)
        assert h * w == x.shape[1]
        x = x.reshape(shape=(x.shape[0], h, w, p, p, 3))
        x = torch.einsum('nhwpqc->nchpwq', x)
        imgs = x.reshape(shape=(x.shape[0], 3, h * p, w * p))
        
        return imgs
    
    def forward_loss(self, imgs, pred, mask):
        target = self.patchify(imgs)
        if self.norm_pix_loss:
            mean = target.mean(dim=-1, keepdim=True)
            var = target.var(dim=-1, keepdim=True)
            target = (target - mean) / (var + 1.e-6)**0.5
        loss = (pred - target) ** 2
        loss = loss.mean(dim=-1)
        loss = (loss * mask).sum() / mask.sum()
        
        return loss
    
    def forward(self, imgs, mask_ratio=None):
        if mask_ratio is None:
            mask_ratio = self.mask_ratio
        latent, mask, ids_restore = self.forward_encoder(imgs, mask_ratio)
        pred = self.forward_decoder(latent, ids_restore)
        loss = self.forward_loss(imgs, pred, mask)
        return loss, pred, mask, ids_restore

In [3]:
class FERClassifier(nn.Module):
    def __init__(self, mae_encoder, num_classes=7):
        super().__init__()
        self.encoder = mae_encoder
        self.classifier = nn.Sequential(
            nn.LayerNorm(mae_encoder.embed_dim),
            nn.Linear(mae_encoder.embed_dim, num_classes)
        )

    def forward(self, x):
        # Patchify & embed: (B, 3, 224, 224) -> (B, N_patches+1, embed_dim)
        tokens = self.encoder.patch_embed(x)
        cls_token = self.encoder.cls_token.expand(x.size(0), -1, -1)
        x = torch.cat((cls_token, tokens), dim=1)
        x = x + self.encoder.pos_embed[:, :x.size(1), :]
        x = self.encoder.pos_drop(x)

        for blk in self.encoder.blocks:
            x = blk(x)

        x = self.encoder.norm(x)
        cls_embedding = x[:, 0]  # Take [CLS] token

        return self.classifier(cls_embedding)

In [4]:
# Set paths
train_dir = "../datasets/FER_2013_enhanced/FER_preprocessed_train"
test_dir = "../datasets/FER_2013_enhanced/preprocessed_test"
# Define the desired class order
desired_class_order = [
    "neutral",   # 0
    "happy",     # 1
    "sad",       # 2
    "surprise",  # 3
    "fear",      # 4
    "disgust",   # 5
    "anger"      # 6
]

# Original FER-2013 class order
original_class_order = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']

# Create a mapping from original index to new index
label_remap = {
    original_class_order.index('neutral'): desired_class_order.index('neutral'),
    original_class_order.index('happy'): desired_class_order.index('happy'),
    original_class_order.index('sad'): desired_class_order.index('sad'),
    original_class_order.index('surprise'): desired_class_order.index('surprise'),
    original_class_order.index('fear'): desired_class_order.index('fear'),
    original_class_order.index('disgust'): desired_class_order.index('disgust'),
    original_class_order.index('angry'): desired_class_order.index('anger')
}

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),  # random crop & scale
    transforms.RandomHorizontalFlip(p=0.5),               # flip horizontally
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),  # color augmentation
    transforms.RandomRotation(15),                        # small rotations
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])
val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),  # deterministic resize for val/test
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

full_train_dataset = datasets.ImageFolder(root=train_dir, transform=train_transform)
test_dataset = datasets.ImageFolder(root=test_dir, transform=val_test_transform)
def remap_labels(dataset):
    # Create new list of samples with remapped labels
    new_samples = []
    for path, label in dataset.samples:
        new_label = label_remap[label]
        new_samples.append((path, new_label))
    
    # Create new dataset with remapped labels
    new_dataset = datasets.ImageFolder(root=dataset.root, transform=dataset.transform)
    new_dataset.samples = new_samples
    new_dataset.targets = [s[1] for s in new_samples]
    new_dataset.classes = desired_class_order
    new_dataset.class_to_idx = {cls: i for i, cls in enumerate(desired_class_order)}
    return new_dataset

# Apply label remapping
full_train_dataset = remap_labels(full_train_dataset)
test_dataset = remap_labels(test_dataset)

# Split training into train and validation
train_size = int(0.8 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size])
# Apply validation transform to the validation set
# val_dataset.dataset = datasets.ImageFolder(train_dir, transform=val_test_transform)
val_dataset.dataset = remap_labels(val_dataset.dataset)
print("Classes:", test_dataset.classes)

Classes: ['neutral', 'happy', 'sad', 'surprise', 'fear', 'disgust', 'anger']


In [5]:
val_dataset.dataset.classes

['neutral', 'happy', 'sad', 'surprise', 'fear', 'disgust', 'anger']

In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
encoder = create_mae_model_from_timm(new_depth=6)
model = SimpleMAE(encoder)
model.load_state_dict(torch.load('../models/model_epoch_35.pth', map_location='cpu'))
model.to(device)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

# Model setup
encoder = model.encoder  # Your pretrained MAE encoder
fer_model = FERClassifier(mae_encoder=encoder, num_classes=7).to(device)
# Freeze encoder initially
def set_encoder_trainable(model, trainable):
    for param in model.encoder.parameters():
        param.requires_grad = trainable

set_encoder_trainable(fer_model, False)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, fer_model.parameters()), lr=1e-4)

# Checkpoint setup
best_val_loss = float('inf')
best_model_path = 'best_fer_model.pth'

# Create optimizer again
optimizer = optim.Adam(filter(lambda p: p.requires_grad, fer_model.parameters()), lr=1e-4)
checkpoint = torch.load('../models/affectNet-7-emotions.pth')  # or best_model_path


fer_model.load_state_dict(checkpoint["model_state_dict"])

✅ Loaded encoder weights.
Missing keys: 0 | Unexpected keys: 72


<All keys matched successfully>

In [ ]:
start_epoch = 0
num_epochs = 15
best_val_loss = float('inf')
label_mapping = {
    4: 0,  # neutral -> neutral
    3: 1,  # happy -> happy
    5: 2,  # sad -> sad
    6: 3,  # surprise -> surprise
    2: 4,  # fear -> fear
    1: 5,  # disgust -> disgust
    0: 6   # anger -> angry
}

for epoch in range(start_epoch, num_epochs):
    if epoch == 6:
        print("🔓 Unfreezing encoder")
        set_encoder_trainable(fer_model, True)
        optimizer = optim.Adam(fer_model.parameters(), lr=1e-5)  # Lower LR for fine-tuning

    # Training phase
    fer_model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        # Create mask on CPU first
        valid_mask = (labels >= 0) & (labels < 7)
        if valid_mask.sum().item() == 0:
            continue  # Skip batch

        # Move to device
        images = images.to(device)
        labels = labels.to(device)
        
        # Apply mask
        images = images[valid_mask]
        labels = labels[valid_mask]

        optimizer.zero_grad()
        outputs = fer_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    train_acc = 100 * correct / total
    print(f"✅ Epoch {epoch+1}: Train Loss={running_loss:.4f}, Train Acc={train_acc:.2f}%")

    # Validation phase
    fer_model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for val_images, val_labels in val_loader:
            # Create mask on CPU first
            valid_mask = (val_labels >= 0) & (val_labels < 7)
            if valid_mask.sum().item() == 0:
                continue

            # Move to device
            val_images = val_images.to(device)
            val_labels = val_labels.to(device)
            
            # Apply mask
            val_images = val_images[valid_mask]
            val_labels = val_labels[valid_mask]

            val_outputs = fer_model(val_images)
            val_loss += criterion(val_outputs, val_labels).item()
            _, val_preds = val_outputs.max(1)
            predicted_mapped = torch.tensor([label_mapping[p.item()] for p in val_preds], 
                                          device=device)
            val_total += val_labels.size(0)
            val_correct += predicted_mapped.eq(val_labels).sum().item()

    val_acc = 100 * val_correct / val_total
    print(f"🔍 Val Loss={val_loss:.4f}, Val Acc={val_acc:.2f}%")

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': fer_model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss,
            'val_acc': val_acc,
        }, '../checkouts/fer_test/best_model.pth')
        print(f"💾 Best model saved at epoch {epoch+1} with val loss {val_loss:.4f}\n")
    else:
        print("")
    
    # Save checkpoint
    torch.save({
        'epoch': epoch + 1,
        'model_state_dict': fer_model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'val_loss': val_loss,
        'val_acc': val_acc,
    }, f'../checkouts/fer_test/fer_model_epoch_{epoch+1}.pth')
    print(f"📝 Checkpoint saved: fer_model_epoch_{epoch+1}.pth\n")

In [ ]:
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)
test_loss = 0.0
test_correct = 0
test_total = 0

with torch.no_grad():  # Disable gradient computation for evaluation
    for test_images, test_labels in test_loader:
        # Move data to the same device as the model
        test_images = test_images.to(device)
        test_labels = test_labels.to(device)
        
        test_outputs = fer_model(test_images)
        test_loss += criterion(test_outputs, test_labels).item()
        
        # Get predictions (corrected: use test_outputs instead of test_output)
        _, test_preds = test_outputs.max(1)  
        
        test_total += test_labels.size(0)
        test_correct += test_preds.eq(test_labels).sum().item()

test_acc = 100 * test_correct / test_total
print(f"✅ Test Accuracy: {test_acc:.2f}%")
print(f"📉 Test Loss: {test_loss:.4f}")